In [140]:
from pathlib import Path
import gzip
import json
import pickle
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
import numpy as np
from stable_platform_matchings.domain.instance import Instance
from stable_platform_matchings.graphs.road_graphs import RoadGraph
import seaborn as sns
from pprint import pprint
from stable_platform_matchings.optimization.optimizer import SolverOptions, OptimizerParams, Optimizer

In [141]:
HET_COST_MEAN = 0.0
HET_COST_SD = 100_000.0

In [142]:
graph_data_filename = "../data/graph_0-14960_00_new.pickle"
instance = Instance.from_yaml(Path('../data/anon_14_day_instances/2020-09-07.yaml'))

with open(graph_data_filename, "rb") as file:
    graph = pickle.load(file)

instance.set_graph(RoadGraph(graph))

all_intermediaries = instance.intermediaries.copy()

In [143]:
high = ["loving_engelbart"]
low = ["competent_mayer"]

In [156]:
instance.intermediaries = [
    intermediary for intermediary in all_intermediaries
    if intermediary.id in high or intermediary.id in low
]

instance.truck_capacity_tons = 50

het_costs = {intermediary.id: float(
        4.0 * instance.dist_to_mill[intermediary.id]
    )
    for intermediary in instance.intermediaries
}

epsilons = {
    intermediary.id: 0 if intermediary.id in low else 1
    for intermediary in instance.intermediaries
}

In [157]:
params = OptimizerParams(
        het_costs=het_costs,
        epsilons=epsilons,
        backend="gurobi",
        vrp_mode="exact",
        vrp_time_limit_seconds=300,
        threads=14
)
optimizer = Optimizer(
    instance=instance,
    params=params,
)

# solve
print("Solving no pay...")
options = SolverOptions(
    strategy="exact",
    structured_farmer_payments=False,
    dominance_constraints=False,
    early_stop_threshold=0.0,
    hist_set_method="instance_farmers",
    pay_unmatched=False,
    stabilize_final_solution=True
)

summary = optimizer.solve(options)



============================= Optimizer Parameters =============================
---------------------------------- het_costs -----------------------------------
  {'competent_mayer': 407088.2819017789, 'loving_engelbart': 1055912.2528911012}
----------------------------------- epsilons -----------------------------------
  {'competent_mayer': 0, 'loving_engelbart': 1}
  backend                    gurobi
  vrp_mode                   exact
  verbose                    True
  print_width                80
  threads                    14
  vrp_time_limit_seconds     300
  farmers                    21
  intermediaries             2


============================== VRP Initialization ==============================
---------------------- Solving for minimum cost matching -----------------------
Set parameter Threads to value 14
Set parameter TimeLimit to value 300
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.3.0 25D2128)

CPU model: Apple M4 Pro
Thread count: 1

In [122]:
N_INTS = 10
SEED = 67

INDO_CRS = "EPSG:23867"
DATA_DIR = Path("../data")

FARMERS_PATH = DATA_DIR / "farmers.csv"
FARMERS_14_PATH = DATA_DIR / "farmers_14.csv"
INTS_PATH = DATA_DIR / "intermediaries.csv"
GRAPH_PATH = DATA_DIR / "graph_0-14960_00_new.pickle"
ALPHA_PATH = DATA_DIR / "precomputed_alpha.json"
SIGMAS_PATH = DATA_DIR / "precomputed_sigmas.json"

In [3]:
ig = InstanceGenerator(
    FARMERS_PATH, FARMERS_14_PATH, INTS_PATH, GRAPH_PATH, ALPHA_PATH, SIGMAS_PATH
)

In [4]:
ig.gen_intermediaries(n_intermediaries=N_INTS, seed=SEED)
ig.gen_calendar(seed=SEED, scale=1.5, n_cycles=10)

In [13]:
platform = ig.gen_instance(instance_id="hello", day=69, n_hist_sets=3)

In [14]:
epsilons = {intermediary.id: 2 for intermediary in platform.intermediaries}
het_costs = {
    intermediary.id: (platform.dist_to_mill[intermediary.id] * 2)
    for intermediary in platform.intermediaries
}

In [15]:
params = OptimizerParams(
    het_costs=het_costs,
    epsilons=epsilons,
    backend="gurobi",
    vrp_mode="approximate",
    verbose=True,
    print_width=80,
    threads=14,
    vrp_time_limit_seconds=3600,
)

opt = Optimizer(platform, params)



============================= Optimizer Parameters =============================
---------------------------------- het_costs -----------------------------------
  {'flamboyant_keldysh': 403773.7090596595,
   'happy_poitras': 671571.7977884224,
   'keen_benz': 371282.1061704554,
   'nervous_roentgen': 316561.2282037674,
   'nostalgic_bohr': 229550.8889446209,
   'nostalgic_dijkstra': 407124.7522323397,
   'suspicious_chaum': 528868.1626196725,
   'suspicious_joliot': 257012.15724020937,
   'vibrant_lumiere': 451893.7016353834,
   'wizardly_elgamal': 514371.7453801243}
----------------------------------- epsilons -----------------------------------
  {'flamboyant_keldysh': 2,
   'happy_poitras': 2,
   'keen_benz': 2,
   'nervous_roentgen': 2,
   'nostalgic_bohr': 2,
   'nostalgic_dijkstra': 2,
   'suspicious_chaum': 2,
   'suspicious_joliot': 2,
   'vibrant_lumiere': 2,
   'wizardly_elgamal': 2}
  backend                    gurobi
  vrp_mode                   approximate
  verbose    

In [16]:
options = SolverOptions(
    strategy="heuristic_optimized",
    structured_farmer_payments=False,
    dominance_constraints=False,
    pay_unmatched=False,
    aggregate=True,
    stabilize_branch_extrema=False,
)


summary = opt.solve(options=options)



================================ Solver Options ================================
  Seed                       0
  Strategy                   heuristic_optimized
  Structured Farmer Payments False
  Dominance Constraints      False
  Early Stop                 False
  Aggregate                  True
  Pay Unmatched              False
  Stabilize Branch Extrema   False


======================== Strategy: heuristic_optimized =========================
  Farmers                    25
  Intermediaries             10


============================== Branch Evaluation ===============================
  Forced matched:
    []
  Forced unmatched:
    []

----------------------------- Primal Solve Result ------------------------------
  Platform profit            1,086,300.326
  Max intermediary welfare   870,107.099
  Max farmer welfare         80,311,700.101
---------------------------- Lower-Bound Candidate -----------------------------
  Objective                  1,086,300.326
  Minimum-co

In [17]:
options = SolverOptions(
    strategy="heuristic_optimized",
    structured_farmer_payments=False,
    dominance_constraints=False,
    pay_unmatched=False,
    aggregate=True,
    stabilize_branch_extrema=True
)


summary = opt.solve(options=options)



================================ Solver Options ================================
  Seed                       0
  Strategy                   heuristic_optimized
  Structured Farmer Payments False
  Dominance Constraints      False
  Early Stop                 False
  Aggregate                  True
  Pay Unmatched              False
  Stabilize Branch Extrema   True


======================== Strategy: heuristic_optimized =========================
  Farmers                    25
  Intermediaries             10


============================== Branch Evaluation ===============================
  Forced matched:
    []
  Forced unmatched:
    []

----------------------------- Primal Solve Result ------------------------------
  Platform profit            1,086,300.326
  Max intermediary welfare   853,149.175
  Max farmer welfare         80,311,700.101
---------------------------- Lower-Bound Candidate -----------------------------
  Objective                  1,086,300.326
  Minimum-cos

In [18]:
options = SolverOptions(
    strategy="exact",
    structured_farmer_payments=False,
    dominance_constraints=False,
    pay_unmatched=False,
    aggregate=True,
    stabilize_branch_extrema=False
)


summary = opt.solve(options=options)



================================ Solver Options ================================
  Seed                       0
  Strategy                   exact
  Structured Farmer Payments False
  Dominance Constraints      False
  Early Stop                 False
  Aggregate                  True
  Pay Unmatched              False
  Stabilize Branch Extrema   False


=============================== Strategy: exact ================================
  Farmers                    25
  Intermediaries             10


============================== Branch Evaluation ===============================
  Forced matched:
    []
  Forced unmatched:
    []

---------------------------- Lower-Bound Candidate -----------------------------
  Objective                  1,086,300.326
  Minimum-cost set:
    ['keen_benz', 'nervous_roentgen', 'nostalgic_bohr', 'suspicious_joliot']

............................. Global Bound Update ..............................
  [STATUS] Improved the global lower bound through forci